# Ray Unit 3 - Sharded Data

Pandas puts an entire DF in RAM. What if the data is too big?

Ray shards the data into blocks while keeping a pandas-like API.

Popular alternatives: Spark, Dask, Daft.


## Setup

Use the same `22971-ray` environment and notebook kernel as the previous units.

In [ ]:
import gc
import math
import time
from pathlib import Path

import numpy as np
import pandas as pd
import ray
from IPython.display import display
from ray.data.aggregate import AggregateFnV2, Count, Mean, Sum
from ray.data.block import BlockAccessor

ray.shutdown()
ray.init(
    ignore_reinit_error=True,
    include_dashboard=False,
    log_to_driver=False,
    logging_level='ERROR',
)
print('Ray version:', ray.__version__)
print('Ray resources:', ray.cluster_resources())

## 0. Generate data

We generate synthetic gameplay telemetry data with these columns:

    | Column | Type |
    |---|---|
    | `event_ts` | `timestamp[ns]` |
    | `player_id` | `int32` |
    | `session_id` | `int32` |
    | `region` | `string` |
    | `platform` | `string` |
    | `event_type` | `string` |
    | `level_id` | `int16` |
    | `latency_ms` | `int32` |
    | `spend_usd` | `float` |
    | `is_test_user` | `bool` |
    | `payload_blob` | `string` |

- `payload_blob` is a long nonsense string that makes each row artificially wide.
- We save the dataset to `...\fake_data\part-XX.parquet`.
- The dataset is split into `NUM_SHARDS` parquet shards.

### Helper functions

In [ ]:
SEED = 0
N_ROWS = 60_000
NUM_SHARDS = 8
PAYLOAD_CHARS = 2_000 #2 KB per row
DATA_DIR = 'fake_data'
OUT_DIR = 'output'

REGIONS = np.array(['NA', 'EU', 'APAC', 'LATAM', 'MEA'])
PLATFORMS = np.array(['pc', 'console', 'mobile'])
EVENT_TYPES = np.array(['match_start', 'match_end', 'quest', 'purchase', 'crash', 'chat'])


def build_payload_column(n_rows, payload_chars=PAYLOAD_CHARS):
    seeds = pd.Series(np.arange(n_rows), dtype='int64').map(
        lambda i: f'blob-{i:07d}-{(i * 2654435761) & 0xFFFFFFFF:08x}'
    )
    return seeds.str.pad(payload_chars, side='right', fillchar='x')


def build_telemetry_frame(n_rows=N_ROWS, seed=SEED, payload_chars=PAYLOAD_CHARS):
    rng = np.random.default_rng(seed)
    event_type = rng.choice(EVENT_TYPES, size=n_rows, p=[0.28, 0.22, 0.25, 0.07, 0.03, 0.15])
    event_ts = pd.Timestamp('2025-01-01') + pd.to_timedelta(
        rng.integers(0, 14 * 24 * 60 * 60, size=n_rows),
        unit='s',
    )
    latency_ms = np.round(
        rng.gamma(shape=2.4, scale=32.0, size=n_rows) + rng.integers(5, 25, size=n_rows)
    ).astype('int32')
    latency_ms[event_type == 'crash'] *= 2

    spend_usd = np.zeros(n_rows, dtype='float32')
    purchase_mask = event_type == 'purchase'
    spend_usd[purchase_mask] = rng.uniform(0.99, 59.99, size=purchase_mask.sum()).round(2)

    frame = pd.DataFrame(
        {
            'event_ts': event_ts,
            'player_id': rng.integers(10_000, 90_000, size=n_rows, dtype=np.int32),
            'session_id': rng.integers(100_000, 900_000, size=n_rows, dtype=np.int32),
            'region': rng.choice(REGIONS, size=n_rows, p=[0.36, 0.27, 0.22, 0.10, 0.05]),
            'platform': rng.choice(PLATFORMS, size=n_rows, p=[0.42, 0.18, 0.40]),
            'event_type': event_type,
            'level_id': rng.integers(1, 60, size=n_rows, dtype=np.int16),
            'latency_ms': latency_ms,
            'spend_usd': spend_usd,
            'is_test_user': rng.random(n_rows) < 0.02,
            'payload_blob': build_payload_column(n_rows, payload_chars),
        }
    )

    return frame.reset_index(drop=True)


def write_parquet_shards(frame, out_dir, num_shards=NUM_SHARDS):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    rows_per_shard = math.ceil(len(frame) / num_shards)
    for shard_id in range(num_shards):
        start = shard_id * rows_per_shard
        stop = min((shard_id + 1) * rows_per_shard, len(frame))
        shard = frame.iloc[start:stop]
        if shard.empty:
            continue
        shard.to_parquet(out_dir / f'part-{shard_id:02d}.parquet', index=False)


def enrich_batch(batch):
    batch = batch.copy()
    batch['hour_of_day'] = batch['event_ts'].dt.hour.astype('int8')
    batch['is_purchase'] = batch['event_type'].eq('purchase')
    batch['is_crash'] = batch['event_type'].eq('crash')
    batch['latency_band'] = pd.cut(
        batch['latency_ms'],
        bins=[-1, 60, 120, 240, np.inf],
        labels=['fast', 'stable', 'slow', 'spiky'],
    ).astype('string')
    return batch

### Data generation

In [ ]:
workspace_dir = Path.cwd()  
input_dir = workspace_dir / DATA_DIR
output_dir = workspace_dir / OUT_DIR

workspace_dir.mkdir(parents=True, exist_ok=True)

existing_shards = sorted(input_dir.glob('part-*.parquet'))
if existing_shards:
    print(f"Found {len(existing_shards)} existing parquet shards in {input_dir}; skipping generation.")
    telemetry_pdf = pd.concat((pd.read_parquet(path) for path in existing_shards), ignore_index=True)
else:
    telemetry_pdf = build_telemetry_frame()
    write_parquet_shards(telemetry_pdf, input_dir, num_shards=NUM_SHARDS)

display(telemetry_pdf.head())
size_bytes = telemetry_pdf.memory_usage(deep=True).sum()
print(f"size: {size_bytes / 1024**2:.2f} MiB")


## 1. Ray Data mental model


#### A Ray `Dataset` feels a lot like a pandas `DataFrame` for the operations in this notebook.

In [ ]:
events_ds = ray.data.read_parquet(str(input_dir), override_num_blocks=NUM_SHARDS)

In [ ]:
events_ds.show(5)

#### The dataset is sharded into **blocks**: each block is stored as a separate object in Ray's object store.

In [ ]:
# We don't need to know what ref_bundles are (ray.data internals)
for bundle_idx, ref_bundle in enumerate(events_ds.iter_internal_ref_bundles()):
    [(block_ref, metadata)] = ref_bundle.blocks
    print(f"block {bundle_idx}: ref={block_ref}, rows={metadata.num_rows}, size={metadata.size_bytes / 1024**2:.2f} MiB")


#### Execution is **lazy**: no data is manipulated unless explicitly required.

In [ ]:
sorted_ds = events_ds.sort(key='event_ts')

Why did this cell run instantly?

Ray Data built a plan:
1. Read the data from the source files, with different tasks reading different blocks.
2. Sort the rows by `event_ts`.

**It did not execute that plan yet.**


In [ ]:
sorted_ds.explain()


We can tell Ray to execute the plan explicitly.

In [ ]:
events_materialized = sorted_ds.materialize()

- Some methods trigger execution (e.g. `show()`, `sum()`, `materialize()`).
- Some methods are lazy (e.g. `sort()`, `groupby()`, `groupby(...).aggregate()`).

If a method returns another `Dataset` or `GroupedData`, it is usually still lazy.

**Common footgun**: triggering execution unintentionally can dominate runtime.

## 2. Cheap vs expensive operations

In pandas, these often feel interchangeable:

```
df.methodA.methodB
df.methodB.methodA
```

With distributed data, order matters much more.

Operations that stay block-local are usually cheap: each block can be processed independently, with no cross-node coordination. These include common ops that are row-independent: `filter`, `select_columns` etc. 

Operations that require global coordination are the expensive ones, especially when they move wide rows across the cluster. Methods like `sort()` and `groupby(...).aggregate(...)` are lazy, but when executed they become expensive global operations.


### Example: wide vs narrow sort

In [ ]:
narrow_columns = [name for name in events_ds.schema().names if name != 'payload_blob']

wide_events = ray.data.read_parquet(str(input_dir), override_num_blocks=NUM_SHARDS)
start = time.perf_counter()
wide_sorted = wide_events.sort('event_ts').materialize()
wide_elapsed = time.perf_counter() - start


In [ ]:

narrow_events = ray.data.read_parquet(str(input_dir), override_num_blocks=NUM_SHARDS)
start = time.perf_counter()
narrow_sorted = (
    narrow_events
    .select_columns(narrow_columns)
    .sort('event_ts')
    .materialize()
)
narrow_elapsed = time.perf_counter() - start

print(f'wide sort elapsed: {wide_elapsed:.2f} s')
print(f'drop payload -> sort elapsed: {narrow_elapsed:.2f} s')
print(f'wide / narrow ratio: {wide_elapsed / narrow_elapsed:.2f}x')
narrow_sorted.show(5)

In [ ]:
print(wide_sorted.stats())
print('\n'+ '='*20+'\n')
print(narrow_sorted.stats())

**Conclusion**: If a later stage does not need a wide column like `payload_blob`, drop it with a block-local op before a global shuffle like `sort()`.

### Example: `groupby` is a communication barrier 

In [ ]:
local_events = ray.data.read_parquet(str(input_dir), override_num_blocks=NUM_SHARDS)
start = time.perf_counter()
local_projected = (
    local_events
    .select_columns(['region', 'latency_ms', 'spend_usd'])
    .materialize()
)
local_elapsed = time.perf_counter() - start

grouped_events = ray.data.read_parquet(str(input_dir), override_num_blocks=NUM_SHARDS)
start = time.perf_counter()
region_summary = (
    grouped_events
    .select_columns(['region', 'latency_ms', 'spend_usd'])
    .groupby('region')
    .aggregate(
        Count(alias_name='events'),
        Mean('latency_ms', alias_name='mean_latency_ms'),
        Sum('spend_usd', alias_name='total_spend_usd'),
    )
    .materialize()
)
grouped_elapsed = time.perf_counter() - start

print(f'project only elapsed: {local_elapsed:.2f} s')
print(f'project -> groupby(region) -> aggregate elapsed: {grouped_elapsed:.2f} s')
print(f'groupby / local ratio: {grouped_elapsed / local_elapsed:.2f}x')
display(region_summary.to_pandas().sort_values('region').reset_index(drop=True))

**Conclusion**: `groupby(...).aggregate(...)` is expensive because Ray cannot finish the job block by block in isolation.

Each block can compute partial results for its own rows, but rows with the same key (e.g. region) may live in different blocks. Before Ray can produce the final answer, it has to move data around so that matching keys end up together.

That extra data movement and synchronization is why `groupby` is a global coordination step.


## 3. Summary

This notebook only covered a high-level slice of Ray Data. The full API is broader, and it also works beyond tabular ETL workloads, including text, images, tensors, and batch inference pipelines.

The key ideas to keep are:
- Think in blocks, not individual rows.
- Use block-local ops like `filter`, `select_columns`, and many `map_batches` transforms to shrink data early.
- Methods like `sort()` and `groupby(...).aggregate(...)` are lazy, but when executed they become expensive global operations.
- `materialize()`, `show()`, `sum()`, `to_pandas()`, and often `count()` are the kinds of calls that force execution.
- If a later stage only needs a few columns, project early before the global op.
- Bring data back to pandas only when the result is genuinely small.

For a broader tour of the API, see the [Ray Data docs](https://docs.ray.io/en/latest/data/key-concepts.html) and the [Aggregating Data guide](https://docs.ray.io/en/latest/data/aggregating-data.html).

## 4. Optional advanced bonus: custom aggregators

You can stop here for the core lesson.

This optional section shows how Ray lets you extend aggregation logic beyond the built-in aggregators.

Custom aggregators follow the same pattern:
1. Look at one block and produce a small partial state.
2. Merge those partial states across blocks.
3. Turn the final state into the answer you want.

Below, the partial state is just `[paid_events, total_events]`, which lets us compute the share of events with non-zero spend in each region.

In [ ]:
class PaidEventRate(AggregateFnV2):
    def __init__(self, on='spend_usd', ignore_nulls=False, alias_name='paid_event_rate'):
        super().__init__(
            alias_name,
            on=on,
            ignore_nulls=ignore_nulls,
            zero_factory=lambda: [0, 0],
        )

    def aggregate_block(self, block):
        pdf = BlockAccessor.for_block(block).to_pandas()
        spend = pdf[self._target_col_name]
        if self._ignore_nulls:
            spend = spend.dropna()
        paid_events = int(spend.gt(0).sum())
        total_events = int(len(spend))
        return [paid_events, total_events]

    def combine(self, current_accumulator, new):
        return [
            current_accumulator[0] + new[0],
            current_accumulator[1] + new[1],
        ]

    def finalize(self, accumulator):
        paid_events, total_events = accumulator
        if total_events == 0:
            return np.nan
        return paid_events / total_events


paid_rate_by_region = (
    events_ds
    .select_columns(['region', 'spend_usd'])
    .groupby('region')
    .aggregate(PaidEventRate(on='spend_usd'))
)

display(paid_rate_by_region.to_pandas().sort_values('region').reset_index(drop=True))